# FIX ASPE

In [19]:
import pandas as pd
import ast

In [20]:
def extract_task_from_path(file_path):
    parts = file_path.split('__')    
    task_name = parts[-1].split('.')[0]
    dataset = parts[-2].split('/')[-1]
    return task_name, dataset


def extract_polarity(aspect_terms_list):
    polarities = []
    for aspect in aspect_terms_list:
        polarities.append(aspect['polarity'])
    return polarities



def extract_terms(aspect_terms_list):
    terms = []
    for aspect in aspect_terms_list:
        terms.append(aspect['term'])
    return terms


def extract_pair(aspect_terms_list):
    pairs = []
    for aspect in aspect_terms_list:
        pairs.append(aspect['term']+':'+aspect['polarity'])
    return pairs


def format_extracted(row):
    try:
        formatted = []
        for item in row:
            # Split only on the first ':' to handle cases where ':' may appear in keys
            key, value = item.strip("'").split("':'", 1)
            formatted.append(f"{key}:{value}")
        return formatted
    except Exception as e:
        return []
    

def get_metrics( y_true, y_pred):
        total_pred = 0
        total_gt = 0
        tp = 0
        for gt, pred in zip(y_true, y_pred):
            gt_list = gt.split(', ')
            pred_list = pred.split(', ')
            total_pred+=len(pred_list)
            total_gt+=len(gt_list)
            for gt_val in gt_list:
                for pred_val in pred_list:
                    if pred_val in gt_val or gt_val in pred_val:
                        tp+=1
                        break
        p = tp/total_pred
        r = tp/total_gt
        if p + r == 0:
            results = 0, 0, 0
            return results
        else:
            return p, r, 2*p*r/(p+r)

def clean_extracted_content(df):
    # Drop rows where 'extracted_content' is NaN
    df = df.dropna(subset=['extracted_content'])
    
    # Strip single quotes from string elements in the list
    df['extracted_content'] = df['extracted_content'].apply(
        lambda items: [item.strip("'") if isinstance(item, str) else item for item in items]
    )
    
    return df

In [35]:
import warnings
warnings.filterwarnings("ignore", category=pd.errors.SettingWithCopyWarning)


def get_metrics_fom_path(path_names):
    for add in path_names:
        try: 
            task_name, dataset = extract_task_from_path(add)
            qwen_result = pd.read_csv(add)
            qwen_result['extracted_content'] = qwen_result['qwen_pred'].str.extract(r"content=[\"'](\[.*?\])[\"']")
            qwen_result['extracted_content'] = qwen_result['extracted_content'].str.strip('[]').str.split(', ')
            qwen_result['aspectTerms'] = qwen_result['aspectTerms'].apply(ast.literal_eval)
            qwen_result = clean_extracted_content(qwen_result)
            if task_name == 'ate':
                qwen_result['label'] = qwen_result['aspectTerms'].apply(extract_terms)
            elif task_name == 'atsc':
                qwen_result['label'] = qwen_result['aspectTerms'].apply(extract_polarity)
                qwen_result = qwen_result[qwen_result['label'] != 'conflict']
            elif task_name == 'aspe':
                qwen_result['label'] = qwen_result['aspectTerms'].apply(extract_pair)
                qwen_result['extracted_content'] = qwen_result['extracted_content'].apply(format_extracted)
            
            precision, recall, f1 = get_metrics(qwen_result['label'].apply(lambda x: ', '.join(x)), qwen_result['extracted_content'].apply(lambda x: ', '.join(x)))

            print(f"Task: {task_name}, Dataset: {dataset}")
            # print(f"Precision: {precision}")   
            # print(f"Recall: {recall}")
            print(f"F1: {f1*100:.2f}")
            print("_" * 50)
        except Exception as e:
            print(f"Error processing {task_name}: {dataset}")
            print("_" * 50)
            continue


In [36]:
path_names = [ 
    '/home/s6moakba/Thesis/agent_practice/qwen_performance/pred/14_l__ate.csv',
    '/home/s6moakba/Thesis/agent_practice/qwen_performance/pred/14_l__atsc.csv',
    '/home/s6moakba/Thesis/agent_practice/qwen_performance/pred/14_l__aspe.csv',

    '/home/s6moakba/Thesis/agent_practice/qwen_performance/pred/14_r__ate.csv',
    '/home/s6moakba/Thesis/agent_practice/qwen_performance/pred/14_r__atsc.csv',
    '/home/s6moakba/Thesis/agent_practice/qwen_performance/pred/14_r__aspe.csv',

    '/home/s6moakba/Thesis/agent_practice/qwen_performance/pred/16__ate.csv',
    '/home/s6moakba/Thesis/agent_practice/qwen_performance/pred/16__atsc.csv',
    '/home/s6moakba/Thesis/agent_practice/qwen_performance/pred/16__aspe.csv',

    '/home/s6moakba/Thesis/agent_practice/qwen_performance/pred/15__ate.csv',
    '/home/s6moakba/Thesis/agent_practice/qwen_performance/pred/15__atsc.csv',
    '/home/s6moakba/Thesis/agent_practice/qwen_performance/pred/15__aspe.csv',
]

In [37]:
get_metrics_fom_path(path_names)

Task: ate, Dataset: 14_l
F1: 60.18
__________________________________________________
Task: atsc, Dataset: 14_l
F1: 65.85
__________________________________________________
Task: aspe, Dataset: 14_l
F1: 43.74
__________________________________________________
Task: ate, Dataset: 14_r
F1: 74.37
__________________________________________________
Task: atsc, Dataset: 14_r
F1: 78.60
__________________________________________________
Task: aspe, Dataset: 14_r
F1: 61.33
__________________________________________________
Task: ate, Dataset: 16
F1: 72.12
__________________________________________________
Task: atsc, Dataset: 16
F1: 78.43
__________________________________________________
Task: aspe, Dataset: 16
F1: 64.07
__________________________________________________
Task: ate, Dataset: 15
F1: 70.12
__________________________________________________
Task: atsc, Dataset: 15
F1: 73.62
__________________________________________________
Task: aspe, Dataset: 15
F1: 58.43
_______________________

# ONE ITERATION

In [149]:
qwen_result = pd.read_csv("/home/s6moakba/Thesis/agent_practice/qwen_performance/pred/14_l__atsc.csv")

In [150]:
qwen_result['extracted_content'] = qwen_result['qwen_pred'].str.extract(r"content=[\"'](\[.*?\])[\"']")

In [151]:
qwen_result['extracted_content'] = qwen_result['extracted_content'].str.strip('[]').str.split(', ')

In [152]:
qwen_result['aspectTerms'] = qwen_result['aspectTerms'].apply(ast.literal_eval)

In [153]:
print(qwen_result['aspectTerms'].iloc[0])

[{'term': 'Boot time', 'polarity': 'positive'}]


In [154]:
# qwen_result['label'] = qwen_result['aspectTerms'].apply(extract_terms)

qwen_result['label'] = qwen_result['aspectTerms'].apply(extract_polarity)

In [157]:
def clean_extracted_content(df):
    # Drop rows where 'extracted_content' is NaN
    df = df.dropna(subset=['extracted_content'])
    
    # Strip single quotes from string elements in the list
    df['extracted_content'] = df['extracted_content'].apply(
        lambda items: [item.strip("'") if isinstance(item, str) else item for item in items]
    )
    
    return df
qwen_result = clean_extracted_content(qwen_result)



/tmp/ipykernel_308128/2751149599.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['extracted_content'] = df['extracted_content'].apply(


In [158]:
roww = qwen_result.iloc[4]
# roww['extracted_content'] = [item.strip("'") for item in roww['extracted_content']]
print(roww['label'])
print(roww['extracted_content'])

['negative', 'negative']
['negative', 'negative']


In [159]:
precision, recall, f1 = get_metrics(qwen_result['label'].apply(lambda x: ', '.join(x)), qwen_result['extracted_content'].apply(lambda x: ', '.join(x)))

In [160]:
print(f"Precision: {precision}")   
print(f"Recall: {recall}")
print(f"F1: {f1}")

Precision: 0.5894105894105894
Recall: 0.5722599418040737
F1: 0.5807086614173228
